1. Minimize the square of the difference of matrix elements between transformed camera and laser images by varying the five (5) 
    parameters involved in the 2D affine transformation.
2. Profit $$$

In [14]:
from skimage import io,transform,util
from scipy.optimize import minimize
import numpy as np

In [29]:
#Read in tif image files.
img_stem=r'C:\Users\jglic\OneDrive - Washington University in St. Louis\Documents\School\WashU\Mukherji Lab\Experiment Images\affine-transf-coords\Beads for alignment - 216nm px size'
camera_in = io.imread(img_stem+r'\yfp-lowlight_1024_fov1.tif')
laser_in = io.imread(img_stem+r'\eyrbow-yellowconfocal_512-fov1.tif')
camera_in=camera_in/np.max(camera_in) #normalize the matrices.
laser_in=laser_in/np.max(laser_in)

In [30]:
# Rescale camera image to new size prior to choosing coords for calculating affine transform.
# path_2048=img_stem+r'\11-7-25 2048_400nm\TRITC_2048_fov1.tif'
def resize_img(path):
    img_raw=io.imread(str(path))
    img_downsc=transform.rescale(img_raw,0.5, anti_aliasing=True)
    img_out = np.zeros((512,512),dtype=float) # hard coded size, okay.
    shape0,shape1 = img_downsc.shape
    img_out[:shape0,:shape1] = img_downsc/np.max(img_downsc) #account for dimension cutoff in full camera fov. Also normalize image here.
    # io.imsave(path[0:-4]+'_downscaled.tif',util.img_as_float32(img_out)) #save edited image as new file with updated name.
    # print("Resized image saved as: "+path[0:-4]+'_downscaled.tif')
    return img_out
# test_down=resize_2048(path_2048) #test output for troubleshooting
camera_in=resize_img(img_stem+r'\yfp-lowlight_1024_fov1.tif')

In [31]:
#Define the 2D affine transformation objective function as a function of its five parameters, 
# and have it return the sum of the squares of the difference in matrices between transformed source and destination.
def afftrans_obj(params):
    sx,sy,tx,ty,shx,shy=params[0],params[1],params[2],params[3],params[5],params[6] #unpack affine transform parameters.
    theta=params[4] #unpack transform.rotate()'s parameter.
    tform=transform.AffineTransform(scale=(sx,sy),translation=(tx,ty),shear=(shx,shy)) #generate the transform with given params.
    camera_rot=transform.rotate(camera_in,theta) #rotate the raw image.
    camera_warped=transform.warp(camera_rot,tform.inverse) #apply the affine transform to the rotated camera image.
    score=np.sum(np.square(laser_in-camera_warped)) #for each param set, assign a score based upon how well it minimizes the difference between warped input and desired output.
    return score


In [32]:
def afftrans(params): #Perform transformation using the optimized parameters.
    sx,sy,tx,ty,shx,shy=params[0],params[1],params[2],params[3],params[5],params[6] #unpack affine transform parameters.
    theta=params[4] #unpack transform.rotate()'s parameter.
    tform=transform.AffineTransform(scale=(sx,sy),translation=(tx,ty),shear=(shx,shy)) #generate the transform with given params.
    camera_rot=transform.rotate(camera_in,theta) #rotate the raw image.
    camera_warped=transform.warp(camera_rot,tform.inverse) #apply the affine transform to the rotated camera image.
    io.imsave(img_stem+r'\yfp-lowlight_1024_fov1_afftransf.tif',util.img_as_float(camera_warped)) #save edited image as new file with updated name.
    return "Sum of transformed matrix = " + str(np.sum(camera_warped))

In [62]:
#Parameter guesses and bounds for optimization function.
g0=np.array([1,  1,  6.5, -20, 87.82, 5e-3, -4e-3]) #parameter set guess.
# g0=np.array([ 9.000e-01,  9.000e-01,  3.007e+00, -2.500e+01,  8.700e+01,
            #  5.000e-02,  1.000e-02])
g0bounds=[(0.98,1.2),(0.98,1.2),(2,8),(-25,-15),(87,88.5),(1e-3,9e-3),(-9e-3,-1e-3)] #bounds on parameter space.
# g0bounds=[(None,None),(None,None),(None,None),(None,None),(None,None),(0,1e-2),(-1e-2,0)] #bounds on parameter space.

In [63]:
soln=minimize(afftrans_obj,x0=g0,bounds=g0bounds,method='L-BFGS-B',tol=1e-15) #run the chosen minimization algorithm.
soln

  message: CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
  success: True
   status: 0
      fun: 2324.1835225071463
        x: [ 1.007e+00  1.008e+00  6.174e+00 -2.254e+01  8.779e+01
             6.487e-03 -5.678e-03]
      nit: 57
      jac: [-3.607e-01 -7.189e-01  9.424e-01 -1.570e-01  2.098e+00
             1.040e+01  1.310e+00]
     nfev: 744
     njev: 93
 hess_inv: <7x7 LbfgsInvHessProduct with dtype=float64>

In [64]:
afftrans(soln['x'])

'Sum of transformed matrix = 24733.30480377148'